In [95]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [96]:
!pip install fastnode2vec

# Lib


In [97]:
import warnings
warnings.filterwarnings('ignore')

In [98]:
import os
import gc
import time
import json
import random
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from fastnode2vec import Graph, Node2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, roc_curve, confusion_matrix, ndcg_score, classification_report

In [99]:
os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)
np.random.seed(42)

# Config

In [100]:
config = {
    # Feature extraction
    'use_node_features': True,
    'use_edge_heuristics': True,
    'use_skill_similarity': True,

    # Node Embedding
    'use_node2vec': True,

    # Column names
    'source_col': 'source',
    'target_col': 'target',
    'label_col': 'label',

    # Path
    "train_path": "/content/drive/MyDrive/colab/Social/train.csv",
    "test_path": "/content/drive/MyDrive/colab/Social/test.csv",
    "feature_json_path": "/content/drive/MyDrive/colab/Social/musae_git_features.json"
}

# Func

## Graph

In [101]:
def build_graph(train_df, test_df, source_col='source', target_col='target', label_col='label'):
    G = nx.Graph()
    train_nodes = set(train_df[source_col].unique()) | set(train_df[target_col].unique())
    test_nodes = set(test_df[source_col].unique()) | set(test_df[target_col].unique())
    all_nodes = train_nodes | test_nodes
    G.add_nodes_from(all_nodes)
    positive_edges = train_df[train_df[label_col] == 1][[source_col, target_col]].values
    G.add_edges_from(positive_edges)
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

## Preprocessing

In [102]:
def preprocessing(X_train, X_test):
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

## Feature extraction

In [103]:
def extract_node_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    deg_cent = nx.degree_centrality(G)
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=500, tol=1e-5)
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality_numpy(G)
    pagerank = nx.pagerank(G, alpha=0.85, tol=1e-5)
    features = pd.DataFrame(index=df.index)
    centralities = [('deg', deg_cent), ('eig', eig_cent), ('pr', pagerank)]
    for name, cent_dict in centralities:
        features[f'{name}_source'] = df[source_col].map(cent_dict)
        features[f'{name}_target'] = df[target_col].map(cent_dict)
    print("[Extract] Completed extract node-level features")
    return features.fillna(0)

In [104]:
def extract_edge_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    features = pd.DataFrame(index=df.index)
    ebunch = list(zip(df[source_col], df[target_col]))
    features['common_neighbors'] = [len(list(nx.common_neighbors(G, u, v))) for u, v in ebunch]
    jac_gen = nx.jaccard_coefficient(G, ebunch)
    features['jaccard'] = [val for _, _, val in jac_gen]
    aa_gen = nx.adamic_adar_index(G, ebunch)
    features['adamic_adar'] = [val for _, _, val in aa_gen]
    def sp_dist(u, v):
        try:
            return nx.shortest_path_length(G, u, v)
        except nx.NetworkXNoPath:
            return len(G.nodes()) + 1
    features['shortest_path'] = [sp_dist(u, v) for u, v in ebunch]
    print("[Extract] Completed extract edge-level features")
    return features

In [105]:
def extract_skill_similarity(df, json_path, source_col='source', target_col='target', use=True, feature_prefix='skill'):
    if not use:
        return pd.DataFrame(index=df.index)
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_dict = json.load(f)
    str_entity_dict = {str(k): set(v) for k, v in raw_dict.items()}
    su_list = [str_entity_dict.get(str(x), set()) for x in df[source_col]]
    sv_list = [str_entity_dict.get(str(x), set()) for x in df[target_col]]
    inter_len = np.array([len(u & v) for u, v in zip(su_list, sv_list)])
    len_u = np.array([len(u) for u in su_list])
    len_v = np.array([len(v) for v in sv_list])
    union_len = len_u + len_v - inter_len
    features = pd.DataFrame(index=df.index)
    col_common = f'{feature_prefix}_common'
    col_jaccard = f'{feature_prefix}_jaccard'
    col_dice = f'{feature_prefix}_dice'
    col_cosine = f'{feature_prefix}_cosine'
    col_overlap = f'{feature_prefix}_overlap_ratio'
    features[col_common] = inter_len
    with np.errstate(divide='ignore', invalid='ignore'):
        features[col_jaccard] = np.where(union_len > 0, inter_len / union_len, 0.0)
        features[col_dice] = np.where((len_u + len_v) > 0, (2.0 * inter_len) / (len_u + len_v), 0.0)
        features[col_cosine] = np.where((len_u * len_v) > 0, inter_len / np.sqrt(len_u * len_v), 0.0)
        min_len = np.minimum(len_u, len_v)
        features[col_overlap] = np.where(min_len > 0, inter_len / min_len, 0.0)
    print(f"[Extract] Completed extract skill similarity")
    return features

## Graph embedding

### Node

In [106]:
def get_node2vec_embeddings(G):
    print("[Embedding] Starting node embedding")
    is_weighted = any("weight" in edge_data for _, _, edge_data in G.edges(data=True))
    if is_weighted:
        edges = [(u, v, data.get("weight", 1.0)) for u, v, data in G.edges(data=True)]
    else:
        edges = list(G.edges())
    n2v_graph = Graph(
        edges,
        directed=G.is_directed(),
        weighted=is_weighted,
        number_of_edges=len(edges),
    )
    DIMENSIONS = 64
    model = Node2Vec(
        n2v_graph,
        dim=DIMENSIONS,
        walk_length=30,
        window=10,
        p=0.5,
        q=2.0,
        workers=4,
        batch_walks=10000,
        seed=42
    )
    model.train(epochs=10)
    actual_dim = model.wv.vector_size if hasattr(model, "wv") else DIMENSIONS
    embeddings = {}
    matched = 0
    missing = 0
    for node in G.nodes():
        if node in model.wv:
            embeddings[node] = np.asarray(model.wv[node], dtype=np.float32)
            matched += 1
        else:
            embeddings[node] = np.zeros(actual_dim, dtype=np.float32)
            missing += 1
    print("[Embedding] Node type:", type(next(iter(G.nodes()))))
    print("[Embedding] First graph nodes:", list(G.nodes())[:10])
    print("[Embedding] First vocab keys:", model.wv.index_to_key[:10])
    print("[Embedding] Embedding dim:", actual_dim)
    print("[Embedding] Matched:", matched)
    print("[Embedding] Missing:", missing)
    if matched > 0:
        sample_node = next(iter(G.nodes()))
        if sample_node in model.wv:
            print("[Embedding] Sample embedding:", model.wv[sample_node][:5])
    print("[Embedding] Completed node embeddings")

    return embeddings

### Edge

In [107]:
def get_edge_embeddings(df, embeddings, operator='hadamard', source_col='source', target_col='target'):
    if embeddings is None:
        return pd.DataFrame(index=df.index)
    dim = len(next(iter(embeddings.values())))
    edge_emb_list = []
    for u, v in zip(df[source_col], df[target_col]):
        emb_u = np.array(embeddings.get(u, np.zeros(dim)))
        emb_v = np.array(embeddings.get(v, np.zeros(dim)))
        if operator == 'hadamard':
            edge_emb = emb_u * emb_v
        elif operator == 'l1':
            edge_emb = np.abs(emb_u - emb_v)
        elif operator == 'l2':
            edge_emb = np.square(emb_u - emb_v)
        elif operator == 'average':
            edge_emb = (emb_u + emb_v) / 2.0
        else:
            raise ValueError("Operator is not allowed")
        edge_emb_list.append(edge_emb)
    columns = [f'n2v_{operator}_{i}' for i in range(dim)]
    print(f"[Embedding] Completed edge embeddings by {operator}")
    return pd.DataFrame(edge_emb_list, columns=columns, index=df.index)

## Train

In [108]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    print(f"[Train] Completed")
    return model

## Eval

In [109]:
def evaluate_global(y_true, y_prob):
    metrics = {
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "AUPR": average_precision_score(y_true, y_prob),
    }
    return pd.DataFrame(metrics.items(), columns=["Metric", "Value"])

def evaluate_ranking_per_node(df_test, y_prob, k_list=[10, 50]):
    df = pd.DataFrame({
        'source': df_test['source'].values,
        'label': df_test['label'].values,
        'prob': y_prob
    })

    metrics_sum = {"MRR": 0.0}
    for k in k_list:
        metrics_sum[f"Precision@{k}"] = 0.0
        metrics_sum[f"Hits@{k}"] = 0.0
        metrics_sum[f"NDCG@{k}"] = 0.0

    num_users = 0
    for source_node, group in df.groupby('source'):
        if group['label'].sum() == 0:
            continue

        num_users += 1

        group = group.sort_values(by='prob', ascending=False)
        labels_sorted = group['label'].values
        probs_sorted = group['prob'].values

        pos_indices = np.where(labels_sorted == 1)[0]
        if len(pos_indices) > 0:
            metrics_sum["MRR"] += 1.0 / (pos_indices[0] + 1)

        for k in k_list:
            top_k_labels = labels_sorted[:k]

            metrics_sum[f"Precision@{k}"] += np.sum(top_k_labels) / k

            metrics_sum[f"Hits@{k}"] += 1 if np.sum(top_k_labels) > 0 else 0

            if len(labels_sorted) > 1:
                ndcg_val = ndcg_score([labels_sorted], [probs_sorted], k=k)
                metrics_sum[f"NDCG@{k}"] += ndcg_val

    final_metrics = []
    for metric_name, total_value in metrics_sum.items():
        mean_value = total_value / num_users if num_users > 0 else 0
        final_metrics.append({"Metric": metric_name, "Value": mean_value})

    return pd.DataFrame(final_metrics)

In [110]:
def visualize_confusion_matrix(y_true, y_probs, threshold=0.5):
    y_pred = (y_probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    sns.kdeplot(y_probs[y_true == 0], fill=True, color="crimson", label="Actual: Label 0", ax=axes[0], alpha=0.5)
    sns.kdeplot(y_probs[y_true == 1], fill=True, color="teal", label="Actual: Label 1", ax=axes[0], alpha=0.5)
    axes[0].axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')
    axes[0].set_title('Predicted Probability Distribution', fontsize=14)
    axes[0].set_xlabel('Predicted Probability of Label 1', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_xlim([0, 1])
    axes[0].legend()

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 14}, ax=axes[1])
    axes[1].set_title(f'Confusion Matrix (Threshold = {threshold})', fontsize=14)
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_xticklabels(['Label 0', 'Label 1'])
    axes[1].set_yticklabels(['Label 0', 'Label 1'])

    plt.tight_layout(pad=0.5)
    plt.show()

## Load

In [111]:
def load_model(model_save_path):
    loaded_model = joblib.load(model_save_path)
    print(f"[Load] Model loaded successfully from {model_save_path}")
    return loaded_model

# Main

## Load data

In [112]:
train_df = pd.read_csv(config['train_path'])
test_df = pd.read_csv(config['test_path'])

## Build graph

In [113]:
G = build_graph(train_df, test_df, source_col=config['source_col'], target_col=config['target_col'], label_col=config['label_col'])

Graph: 37700 nodes, 231203 edges


## Feature extraction

In [114]:
node_train = extract_node_level_features(G, train_df, config['source_col'], config['target_col'], config['use_node_features'])
edge_train = extract_edge_level_features(G, train_df, config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_train = extract_skill_similarity(train_df, config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

[Extract] Completed extract node-level features
[Extract] Completed extract edge-level features
[Extract] Completed extract skill similarity


In [115]:
node_test  = extract_node_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_node_features'])
edge_test  = extract_edge_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_test  = extract_skill_similarity(test_df,  config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

[Extract] Completed extract node-level features
[Extract] Completed extract edge-level features
[Extract] Completed extract skill similarity


In [116]:
if config['use_node_features']:
  node_train, node_test, _ = preprocessing(node_train, node_test)
  node_train = pd.DataFrame(node_train, index=train_df.index)
  node_test = pd.DataFrame(node_test, index=test_df.index)

if config['use_edge_heuristics']:
  edge_train, edge_test, _ = preprocessing(edge_train, edge_test)
  edge_train = pd.DataFrame(edge_train, index=train_df.index)
  edge_test  = pd.DataFrame(edge_test,  index=test_df.index)

if config['use_skill_similarity']:
  skill_train, skill_test, _ = preprocessing(skill_train, skill_test)
  skill_train = pd.DataFrame(skill_train, index=train_df.index)
  skill_test  = pd.DataFrame(skill_test,  index=test_df.index)

## Node embeddings

In [117]:
embeddings = None
if config['use_node2vec']:
    embeddings = get_node2vec_embeddings(G)

[Embedding] Starting node embedding


Reading graph:   0%|          | 0/231203 [00:00<?, ?it/s]

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

[Embedding] Node type: <class 'numpy.int64'>
[Embedding] First graph nodes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
[Embedding] First vocab keys: [np.int64(37484), np.int64(37409), np.int64(37485), np.int64(37239), np.int64(37472), np.int64(37545), np.int64(37569), np.int64(36666), np.int64(36966), np.int64(36429)]
[Embedding] Embedding dim: 64
[Embedding] Matched: 37699
[Embedding] Missing: 1
[Embedding] Sample embedding: [ 0.00279497  0.13099505 -0.06568747 -0.36358988 -0.03851237]
[Embedding] Completed node embeddings


## Edge embeddings

In [118]:
operator = 'hadamard'
n2v_train = get_edge_embeddings(df=train_df, embeddings=embeddings, operator=operator, source_col=config['source_col'], target_col=config['target_col'])
n2v_test  = get_edge_embeddings(df=test_df, embeddings=embeddings, operator=operator, source_col=config['source_col'], target_col=config['target_col'])

[Embedding] Completed edge embeddings by hadamard
[Embedding] Completed edge embeddings by hadamard


## Matrix features

In [119]:
X_train = None
X_test = None
X_train = pd.concat([node_train, edge_train, skill_train, n2v_train], axis=1)
X_test  = pd.concat([node_test,  edge_test,  skill_test,  n2v_test],  axis=1)

In [120]:
X_train = X_train.values
X_test = X_test.values
y_train = train_df[config['label_col']]
y_test  = test_df[config['label_col']]

## Model

In [121]:
svm_model = load_model('/content/drive/MyDrive/colab/Social/model/svm.joblib')

[Load] Model loaded successfully from /content/drive/MyDrive/colab/Social/model/svm.joblib


In [122]:
log_model = load_model('/content/drive/MyDrive/colab/Social/model/log.joblib')

[Load] Model loaded successfully from /content/drive/MyDrive/colab/Social/model/log.joblib


In [123]:
rf_model = load_model('/content/drive/MyDrive/colab/Social/model/random_forest.joblib')

[Load] Model loaded successfully from /content/drive/MyDrive/colab/Social/model/random_forest.joblib


In [124]:
xgb_model = load_model('/content/drive/MyDrive/colab/Social/model/xgb.joblib')

[Load] Model loaded successfully from /content/drive/MyDrive/colab/Social/model/xgb.joblib


## Eval

In [125]:
y_pred_rf = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

y_pred_svm = svm_model.predict(X_test)
y_prob_svm = svm_model.decision_function(X_test)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

In [126]:
rf_df = evaluate_global(y_test, y_prob_rf)
rf_df = rf_df.set_index('Metric').T
rf_df.index = ['Random Forest']

xgb_df = evaluate_global(y_test, y_prob_xgb)
xgb_df = xgb_df.set_index('Metric').T
xgb_df.index = ['XGBoost']

svm_df = evaluate_global(y_test, y_prob_svm)
svm_df = svm_df.set_index('Metric').T
svm_df.index = ['SVM']

log_df = evaluate_global(y_test, y_prob_log)
log_df = log_df.set_index('Metric').T
log_df.index = ['Logistic Regression']

combined_df = pd.concat([rf_df, xgb_df, svm_df, log_df])
combined_df.round(4).sort_values(by='AUPR', ascending=False)

Metric,ROC-AUC,AUPR
XGBoost,0.9512,0.9578
Random Forest,0.9495,0.9562
Logistic Regression,0.9284,0.9400
SVM,0.9248,0.9374


In [127]:
y_prob_svm = svm_model.decision_function(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

rf_ranking_df = evaluate_ranking_per_node(test_df, y_prob_rf)
rf_ranking_df = rf_ranking_df.set_index('Metric').T
rf_ranking_df.index = ['Random Forest']

xgb_ranking_df = evaluate_ranking_per_node(test_df, y_prob_xgb)
xgb_ranking_df = xgb_ranking_df.set_index('Metric').T
xgb_ranking_df.index = ['XGBoost']

svm_ranking_df = evaluate_ranking_per_node(test_df, y_prob_svm)
svm_ranking_df = svm_ranking_df.set_index('Metric').T
svm_ranking_df.index = ['SVM']

log_ranking_df = evaluate_ranking_per_node(test_df, y_prob_log)
log_ranking_df = log_ranking_df.set_index('Metric').T
log_ranking_df.index = ['Logistic Regression']

combined_ranking_df = pd.concat([rf_ranking_df, xgb_ranking_df, svm_ranking_df, log_ranking_df])
combined_ranking_df.round(4)

Metric,MRR,Precision@10,Hits@10,NDCG@10,Precision@50,Hits@50,NDCG@50
Random Forest,0.9649,0.2657,1.0,0.8473,0.0634,1.0,0.8481
XGBoost,0.9672,0.2657,1.0,0.8490,0.0634,1.0,0.8499
SVM,0.9592,0.2654,1.0,0.8419,0.0634,1.0,0.8429
Logistic Regression,0.9616,0.2654,1.0,0.8437,0.0634,1.0,0.8447


In [128]:
df_result = combined_df.merge(combined_ranking_df, left_index=True, right_index=True)
df_result.round(4).sort_values(by='AUPR', ascending=False)

Metric,ROC-AUC,AUPR,MRR,Precision@10,Hits@10,NDCG@10,Precision@50,Hits@50,NDCG@50
XGBoost,0.9512,0.9578,0.9672,0.2657,1.0,0.8490,0.0634,1.0,0.8499
Random Forest,0.9495,0.9562,0.9649,0.2657,1.0,0.8473,0.0634,1.0,0.8481
Logistic Regression,0.9284,0.9400,0.9616,0.2654,1.0,0.8437,0.0634,1.0,0.8447
SVM,0.9248,0.9374,0.9592,0.2654,1.0,0.8419,0.0634,1.0,0.8429
